# Monte Carlo Tree Search — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
DIRS=[(1,0),(-1,0),(0,1),(0,-1)]
def neighbors(p, shape=(5,5), walls=set()):
    for dr,dc in DIRS:
        q=(p[0]+dr,p[1]+dc)
        if 0<=q[0]<shape[0] and 0<=q[1]<shape[1] and q not in walls: yield q
def reconstruct(parent, goal):
    path=[]; x=goal
    while x is not None: path.append(x); x=parent.get(x)
    return path[::-1]
def draw_grid(path=(), explored=(), frontier=(), walls=set(), start=(0,0), goal=(4,4), shape=(5,5), title='grid'):
    img=np.zeros(shape)
    for r,c in walls: img[r,c]=-1
    for r,c in explored: img[r,c]=.35
    for r,c in frontier: img[r,c]=.65
    for r,c in path: img[r,c]=1
    plt.figure(figsize=(4,4)); plt.imshow(img,cmap='viridis',vmin=-1,vmax=1); plt.scatter([start[1]],[start[0]],c='w',edgecolor='k'); plt.scatter([goal[1]],[goal[0]],c='r',edgecolor='k'); plt.title(title); plt.xticks(range(shape[1])); plt.yticks(range(shape[0])); plt.grid(color='w',alpha=.3)
print('setup ok')

## Simulate to allocate search effort
MCTS combines average reward with an exploration bonus and backs rollout results up the tree.

In [ ]:
N=20; visits=np.array([10,2,8]); wins=np.array([6,1,5]); ucb=wins/visits+1.4*np.sqrt(np.log(N)/visits); plt.figure(figsize=(5,3)); plt.bar(['A','B','C'],ucb); plt.title('UCB')
assert int(np.argmax(ucb))==1 and round(float(ucb[1]),3)==2.213

In [ ]:
visits=np.array([10.,2.,8.]); wins=np.array([6.,1.,5.]); ucb=wins/visits+1.4*np.sqrt(np.log(20)/visits); choice=int(np.argmax(ucb)); visits[choice]+=1; wins[choice]+=1; means=wins/visits; plt.figure(figsize=(5,3)); plt.bar(['A','B','C'],means); plt.title('backup')
assert visits[1]==3 and round(float(means[1]),3)==0.667

In [ ]:
rng=np.random.default_rng(2); true=[.55,.70,.45]; samples=[rng.binomial(1,p,30).mean() for p in true]; plt.figure(figsize=(5,3)); plt.bar(['A','B','C'],samples); plt.title('rollouts')
assert int(np.argmax(samples))==1

In [ ]:
rng=np.random.default_rng(3); true=[.55,.70,.45]; wins=np.array([1.,0.,1.]); visits=np.ones(3)
for t in range(3,103):
 a=int(np.argmax(wins/visits+1.4*np.sqrt(np.log(t)/visits))); wins[a]+=rng.random()<true[a]; visits[a]+=1
plt.figure(figsize=(5,3)); plt.bar(['A','B','C'],visits); plt.title('visit allocation')
assert int(np.argmax(visits))==1 and visits.sum()==103

In [ ]:
means=wins/visits; plt.figure(figsize=(5,3)); x=np.arange(3); plt.bar(x-.15,visits,.3); plt.bar(x+.15,means*100,.3); plt.title('final choice')
assert int(np.argmax(visits))==1 and means[1]>means[2]